# 03. The mechanism: why a mortgage hedge goes stale

**The question.** How does the borrower's refinancing option make an MBS's rate exposure move, and what happens to a hedge that is not adjusted?

**Why a portfolio manager cares.** This is the reason mortgage desks re-hedge. A manager who understands the mechanism knows when exposure is likely to move fast (near the money) and when it should be stable (deep in or out of the money).

**Learning objectives**

1. Compare fixed Treasury cash flows with mortgage cash flows that shorten through prepayment.
2. See how an assumed prepayment curve turns into a duration that rises with yields.
3. Derive negative convexity from the identity C = D² − dD/dy.
4. Quantify the loss of a hedge sized once and never adjusted.
5. Explore the development-period data (through 2014) only.

| | |
|---|---|
| Inputs | `teaching.py` (no data) and, for Part D, the daily panel from notebook 02 |
| Outputs | figures `03_*` in `reports/figures/`, `results/teaching/*.csv` |
| Run after | 02 (only for Part D) |
| Approximate runtime | under a minute |

> **Parts A to C use assumed behavior, not data.** The prepayment curve, spreads, and pool terms are chosen for clarity. They illustrate the mechanism. Estimated historical evidence starts in Part D and in notebook 05.

In [ ]:
# Setup: find the project root, load the configuration, and pick the run mode.
# The mode comes from MBSHEDGE_MODE (set by run_project.py) or config/project.yaml.
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "config" / "project.yaml").exists())
sys.path.insert(0, str(ROOT / "src"))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from mbshedge.config import setup, experiment
from mbshedge import plotting as pt

cfg, paths, MODE = setup()
EXP = experiment(cfg, MODE)
pt.setup_style()
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)

In [ ]:
from mbshedge import teaching as tc
from mbshedge import instruments as ins
tdir = paths.results / "teaching"; tdir.mkdir(parents=True, exist_ok=True)

## Part A. Cash flows: fixed versus prepayable

A 10-year Treasury pays the same coupons and principal whatever happens to rates. A mortgage pool's principal arrives in three ways: scheduled amortization, home sales, and refinancing. The faster borrowers prepay (the higher the CPR, the annual share of the remaining balance repaid early), the sooner the principal comes back.

The pool: $100 of 30-year loans at a 6.5% note rate, passing a 6.0% coupon to investors.

In [ ]:
rows = []
for cpr in [0.06, 0.15, 0.30, 0.50]:
    cf = tc.pool_cashflows(cpr)
    m = tc.cashflow_measures(cf, 0.06)
    rows.append({"CPR (annual prepayment rate)": f"{cpr:.0%}", "weighted-average life (years)": round(m["wal_years"], 2),
                 "duration of the cash flows (years)": round(m["duration"], 2),
                 "principal returned in first 3 years ($)": round((cf.loc[cf["month"] <= 36, "scheduled"] + cf.loc[cf["month"] <= 36, "prepaid"]).sum(), 1)})
partA = pd.DataFrame(rows); partA.to_csv(tdir / "cashflow_table.csv", index=False)
partA

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8))
for cpr, col in [(0.06, "#2a78d6"), (0.30, "#eb6834"), (0.50, "#1baf7a")]:
    cf = tc.pool_cashflows(cpr)
    bal = cf.set_index("month")["balance_start"]
    ax.plot(bal.index / 12, bal, color=col, label=f"mortgage pool, CPR {cpr:.0%}")
ax.plot([0, 10, 10], [100, 100, 0], color=pt.INK, lw=1.4, ls="--", label="10-year Treasury (principal at maturity)")
ax.set_xlabel("years"); ax.set_ylabel("principal outstanding ($ per $100)"); ax.set_xlim(0, 30)
ax.set_title("Prepayment returns mortgage principal early")
ax.legend()
pt.save_figure(fig, "03_principal_outstanding", paths, mode=MODE)
plt.show()

## Part B. From the refinancing incentive to duration and negative convexity

Assume borrowers prepay along an S-curve in the incentive (note rate minus the current mortgage rate): about 6% a year from home sales when refinancing does not pay, rising toward 51% when it saves well over one point. In a scenario where the 10-year Treasury yield is $y$, the current mortgage rate is $y + 1.7\%$, so falling yields raise the incentive and the CPR.

Use the duration of the projected cash flows, $D(y)$, as the pool's price sensitivity, and build the price curve from it:

$$
\ln P(y) - \ln P(y_0) = -\int_{y_0}^{y} D(u)\, du .
$$

Differentiating twice gives convexity:

$$
C(y) = \frac{1}{P}\frac{d^2 P}{dy^2} = D(y)^2 - \frac{dD}{dy}.
$$

For a Treasury, duration barely changes with yield, so $C \approx D^2 > 0$. For the mortgage pool near the money, duration rises steeply with yields ($dD/dy$ large), so $C$ turns **negative**: the price rises less in a rally than it falls in a selloff.

In [ ]:
grid = np.round(np.arange(0.025, 0.0751, 0.0025), 5)
scen = tc.scenario_table(grid)
scen.to_csv(tdir / "scenario_table.csv", index=False)
scen.round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
x = scen["treasury_yield_pct"]
axes[0].plot(x, scen["treasury_price_index"], color=pt.INK, ls="--", label="10-year Treasury")
axes[0].plot(x, scen["mbs_price_index"], color="#1baf7a", label="Mortgage pool")
axes[0].set_title("Price (100 at 4.8%)"); axes[0].set_xlabel("10-year yield (%)"); axes[0].legend()
axes[1].plot(x, scen["treasury_duration"], color=pt.INK, ls="--")
axes[1].plot(x, scen["mbs_duration"], color="#1baf7a")
axes[1].set_title("Duration (years)"); axes[1].set_xlabel("10-year yield (%)")
axes[2].plot(x, scen["treasury_convexity"], color=pt.INK, ls="--")
axes[2].plot(x, scen["mbs_convexity"], color="#1baf7a")
axes[2].axhline(0, color=pt.INK_2, lw=0.8)
axes[2].set_title("Convexity C = D^2 - dD/dy (years^2)"); axes[2].set_xlabel("10-year yield (%)")
pt.save_figure(fig, "03_negative_convexity", paths, data=scen, mode=MODE)
plt.show()

## Part C. A hedge sized once and never adjusted

At a 4.8% Treasury yield, size a 10-year Treasury short so its DV01 matches the pool's. Then move all yields instantly and keep the hedge unchanged.

In [ ]:
stale = tc.stale_hedge_pnl(np.array([-200, -100, -50, -25, 0, 25, 50, 100, 200]))
stale.to_csv(tdir / "stale_hedge.csv", index=False)
print(f"Initial duration {stale.attrs['initial_duration']:.2f} years -> hedge ratio {stale.attrs['initial_hedge_ratio']:.3f} "
      "($ of 10-year Treasury sold per $1 of MBS)")
(stale[["shift_bp", "mbs_pnl_usd", "hedge_pnl_usd", "hedged_pnl_usd", "mbs_duration_after", "required_hedge_ratio_after"]]
 .assign(**{c: lambda d, c=c: (d[c] / 1e6).round(2) for c in ["mbs_pnl_usd", "hedge_pnl_usd", "hedged_pnl_usd"]})
 .rename(columns={"mbs_pnl_usd": "MBS P&L ($mm)", "hedge_pnl_usd": "hedge P&L ($mm)", "hedged_pnl_usd": "hedged P&L ($mm)"}).round(3))

In [ ]:
loss_dn = float(stale.loc[stale.shift_bp == -100, "hedged_pnl_usd"].iloc[0]) / 1e6
loss_up = float(stale.loc[stale.shift_bp == 100, "hedged_pnl_usd"].iloc[0]) / 1e6
print(f"In this stylized example a $100mm position hedged once loses ${-loss_dn:.2f}mm if yields fall 100 bp and "
      f"${-loss_up:.2f}mm if they rise 100 bp: the hedged position is short convexity.")
print("Re-hedging after each move resets the hedge ratio, but it trades after the move, so it locks in part of the loss. "
      "That cost is what the mortgage spread pays the investor to bear.")

## Part D. Development-period exploration (data through December 2014 only)

The test period stays closed here. The plots below use only months up to the end of development, so nothing in them can influence the frozen choices. Months before mid-2013 rely on the retrospective proxy.

An exponentially weighted regression duration (63-trading-day half-life, using data up to each date) is shown for intuition only. Neighboring points share most of their data, so no inference is drawn from it.

In [ ]:
from mbshedge import exposure as ex
daily = pd.read_parquet(paths.processed / "daily_panel.parquet")
dev_end = pd.Period(EXP["sample"]["development_end"], "M").to_timestamp(how="end")
dev = daily.loc[:dev_end]
roll = ex.ewma_duration(dev, 63)["U"]        # 63-trading-day half-life, data up to each date
fig, axes = plt.subplots(2, 1, figsize=(10, 5.5), sharex=True)
axes[0].plot(roll.index, roll, color="#eb6834", lw=1.4, label="EWMA regression duration (63-day half-life)")
axes[0].set_ylabel("years"); axes[0].set_title("Development period: empirical duration of MBB")
axes[0].legend(loc="upper left")
axes[1].plot(dev.index, dev["g"], color="#1baf7a", lw=1.4, label="share of borrowers in the money")
axes[1].set_ylabel("share"); axes[1].set_ylim(0, 1); axes[1].legend(loc="upper left")
for a in axes:
    a.axvspan(dev.index.min(), dev.index[dev["incentive_source"] == "nmdb"].min() if (dev["incentive_source"] == "nmdb").any() else dev.index.max(), color=pt.EPISODE_FILL, lw=0)
    pt.shade_episodes(a, [e for e in EXP["episodes"] if e["period"] == "development"])
axes[1].text(dev.index.min(), 0.97, " shaded background: retrospective proxy period", transform=axes[1].get_xaxis_transform(), va="top", fontsize=8, color=pt.INK_2)
pt.save_figure(fig, "03_development_exploration", paths, data=pd.DataFrame({"ewma_duration_63d": roll, "g": dev["g"]}), mode=MODE)
plt.show()
corr = pd.concat([roll, dev["g"]], axis=1).dropna().resample("ME").last().corr().iloc[0, 1]
print(f"Development-period correlation of month-end EWMA duration with the share in the money: {corr:+.2f} "
      "(descriptive, retrospective before mid-2013)")